This notebook documents the workflow for preparing geospatial land use data for Latvia. 
It covers the download, extraction, and processing of multiple open datasets, including agricultural parcels (LAD), forest stands (MVR), biotope polygons, artificial points (address points and bus stops). The script harmonizes these sources, classifies land use types, generates representative sample points for each class, and ensures spatial exclusivity for training and validation in further geospatial analyses or machine learning tasks.

In [1]:
import os
import requests
from datetime import datetime
import py7zr
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape


#### LAD data download

In [ ]:
# define URLs for LAD agricultural parcels data
austrumlatgale_LAD_lauki_url = "https://data.gov.lv/dati/dataset/33c06a35-2234-4982-a5c9-95925076cf64/resource/1a146bdb-4004-44cd-bd72-d118a28529c8/download/austrumlatgale.gpkg"
dienvidkurzeme_LAD_lauki_url = "https://data.gov.lv/dati/dataset/33c06a35-2234-4982-a5c9-95925076cf64/resource/3b8817cf-c894-441b-8d6f-a64a8481e689/download/dienvidkurzeme.gpkg"
dienvidlatgale_LAD_lauki_url = "https://data.gov.lv/dati/dataset/33c06a35-2234-4982-a5c9-95925076cf64/resource/dbbf58f6-5564-40ea-b282-fcaf87729be0/download/dienvidlatgale.gpkg"
lielriga_LAD_lauki_url = "https://data.gov.lv/dati/dataset/33c06a35-2234-4982-a5c9-95925076cf64/resource/e77e3e87-720b-4266-8741-711e20002de6/download/lielriga.gpkg"
viduslatvija_LAD_lauki_url = "https://data.gov.lv/dati/dataset/33c06a35-2234-4982-a5c9-95925076cf64/resource/fae646aa-08fd-44a5-8684-5f96d97b832f/download/viduslatvija.gpkg"
zemgale_LAD_lauki_url = "https://data.gov.lv/dati/dataset/33c06a35-2234-4982-a5c9-95925076cf64/resource/a116c40b-6ef8-464f-b6ac-e525e123f151/download/zemgale.gpkg"
ziemelaustrumi_LAD_lauki_url = "https://data.gov.lv/dati/dataset/33c06a35-2234-4982-a5c9-95925076cf64/resource/5295e49c-a091-431b-8b97-e17971e1887a/download/ziemelaustrumi.gpkg"
ziemelkurzeme_LAD_lauki_url = "https://data.gov.lv/dati/dataset/33c06a35-2234-4982-a5c9-95925076cf64/resource/01f82a0b-0b52-43a0-bae4-1f32ead66671/download/ziemelkurzeme.gpkg"
ziemelvidzeme_LAD_lauki_url = "https://data.gov.lv/dati/dataset/33c06a35-2234-4982-a5c9-95925076cf64/resource/1f278440-7bfc-4b29-9ce1-111ed91756ec/download/ziemelvidzeme.gpkg"

In [ ]:
# download and read LAD agricultural parcels data
LAD_lauki_url_list = [
    austrumlatgale_LAD_lauki_url,
    dienvidkurzeme_LAD_lauki_url,
    dienvidlatgale_LAD_lauki_url,
    lielriga_LAD_lauki_url,
    viduslatvija_LAD_lauki_url,
    zemgale_LAD_lauki_url,
    ziemelaustrumi_LAD_lauki_url,
    ziemelkurzeme_LAD_lauki_url,
    ziemelvidzeme_LAD_lauki_url
]

for region_url in LAD_lauki_url_list:
    region_gdf = gpd.read_file(region_url)
    if 'lad_gdf' in locals():
        lad_gdf = pd.concat([lad_gdf, region_gdf], ignore_index=True)
    else:
        lad_gdf = region_gdf

lad_gdf.to_file("../data/combined_lauki.gpkg", driver="GPKG")

#### MVR data download

In [ ]:
# define URLs for MVR forest stand data
austrumu_MVR_url = "https://data.gov.lv/dati/dataset/40014c0a-90f5-42be-afb2-fe3c4b8adf92/resource/584ce572-e3f9-4fb4-94a6-7fc8c4c34330/download/austrumu.7z"
centra_MVR_url = "https://data.gov.lv/dati/dataset/40014c0a-90f5-42be-afb2-fe3c4b8adf92/resource/392dfb67-eeeb-43c2-b082-35f9cf986128/download/centra.7z"
dienvidu_MVR_url = "https://data.gov.lv/dati/dataset/40014c0a-90f5-42be-afb2-fe3c4b8adf92/resource/bbf4820e-4b77-4563-bc12-dc95df1cec60/download/dienvidu.7z"
kurzemes_MVR_url = "https://data.gov.lv/dati/dataset/40014c0a-90f5-42be-afb2-fe3c4b8adf92/resource/5019d0a9-cf54-4509-8753-d4c159a872a8/download/kurzemes.7z"
vidzemes_MVR_url = "https://data.gov.lv/dati/dataset/40014c0a-90f5-42be-afb2-fe3c4b8adf92/resource/9a755463-7271-41a1-9ee6-fce76e68e146/download/vidzemes.7z"

data_dir = "../data/mvr"

In [ ]:
# download and extract the data
os.makedirs(data_dir, exist_ok=True)

MVR_URL_list = [austrumu_MVR_url, centra_MVR_url, dienvidu_MVR_url, kurzemes_MVR_url, vidzemes_MVR_url]

for region_url in MVR_URL_list:
    region = region_url.split("/")[-1].split(".")[0]  
    archive_path = os.path.join(data_dir, f"mvr_{region}.7z")
    extract_path = os.path.join(data_dir, f"mvr_{region}")
    os.makedirs(extract_path, exist_ok=True)

    response = requests.get(region_url)
    response.raise_for_status()

    with open(archive_path, 'wb') as f:
        f.write(response.content)

    with py7zr.SevenZipFile(archive_path, mode='r') as archive:
        archive.extractall(path=extract_path)

In [ ]:
# collect all shapefiles from the extracted MVR data, save the data
shapefiles = []
for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.endswith(".shp"):
            shapefiles.append(os.path.join(root, file))

print(f"{len(shapefiles)} shapefiles.")

gdf_list = []
for shp in shapefiles:
    try:
        gdf = gpd.read_file(shp)
        gdf_list.append(gdf)
    except Exception as e:
        print(f"Error with {shp}: {e}")

if gdf_list:
    mvr_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True), crs=gdf_list[0].crs)
    print("Combined.")
else:
    mvr_gdf = gpd.GeoDataFrame()
    print("No polygons.")


mvr_gdf.to_file("../data/combined_mezi.gpkg", driver="GPKG")

#### Biotope data download

In [ ]:
# define URL for biotope data, download and save the data
biotopi_url = "https://data.gov.lv/dati/dataset/7b5ac3e9-aaa5-44d6-a828-777736edeb71/resource/e52b8094-f563-4706-b59f-401f7113a133/download/biotopi.zip"

In [ ]:
# download and save biotope data
biotopi_req = requests.get(biotopi_url)
with open("../data/biotopi.zip", 'wb') as f:
    f.write(biotopi_req.content)

#### Artificial land data point download

In [ ]:
# define URL for artificial surfaces data, download and save the data
adreses_url = "https://data.gov.lv/dati/dataset/6b06a7e8-dedf-4705-a47b-2a7c51177473/resource/a510737a-18ce-400f-ad4b-04fce5228272/download/aw_eka.csv"
pieturas_url = "https://data.gov.lv/dati/dataset/ca340e22-3633-473d-94a3-3919bc68e98d/resource/b754ac23-0675-492d-aa64-1f0d7da218c7/download/busstops.csv"

adreses_df = pd.read_csv(adreses_url, encoding="utf-8", sep=",", low_memory=False)
adreses_gdf = gpd.GeoDataFrame(adreses_df, geometry=gpd.points_from_xy(adreses_df["KOORD_Y"], adreses_df["KOORD_X"]), crs="EPSG:3059")
adreses_gdf.to_file("../data/adreses.gpkg", driver="GPKG")

pieturas_df = pd.read_csv(pieturas_url, encoding="utf-8", sep=",", low_memory=False)
pieturas_gdf = gpd.GeoDataFrame(pieturas_df, geometry=gpd.points_from_xy(pieturas_df["Longitude"], pieturas_df["Latitude"]), crs="EPSG:4326")
pieturas_gdf = pieturas_gdf.to_crs("EPSG:3059")
pieturas_gdf.to_file("../data/pieturas.gpkg", driver="GPKG")

#### OSM data download (optional: restore relevant code)

In [ ]:
# # define functions for Overpass API querying and OSM to GeoDataFrame conversion
# def query_overpass(query: str) -> dict:
#     url = "https://overpass-api.de/api/interpreter"
#     response = requests.get(url, params={'data': query})
#     response.raise_for_status()
#     return response.json()

# def osm_to_geodf(osm_json: dict, landclass: str) -> gpd.GeoDataFrame:
#     features = []
#     for el in osm_json.get("elements", []):
#         if el["type"] not in ("way", "relation"):
#             continue
#         if "geometry" not in el:
#             continue

#         coords = [(pt["lon"], pt["lat"]) for pt in el["geometry"]]
#         if coords[0] != coords[-1]:
#             coords.append(coords[0])

#         try:
#             geom = shape({"type": "Polygon", "coordinates": [coords]})
#         except Exception:
#             continue

#         features.append({
#             "id": el.get("id"),
#             "land_use_type": landclass,
#             **el.get("tags", {}),
#             "geometry": geom
#         })

#     if not features:
#         return gpd.GeoDataFrame(columns=["id", "land_use_type", "geometry"], geometry="geometry", crs="EPSG:4326")

#     gdf = gpd.GeoDataFrame(features, geometry="geometry", crs="EPSG:4326")
#     gdf = gdf[gdf.is_valid]
#     return gdf


# # define Overpass API query header for Latvia
# header = """
#     [out:json][timeout:600];
#     area["int_name"="Latvia"][admin_level=2]->.searchArea;
#     """
    
# # define Overpass API queries for different land use classes
# classes = {
#     "artificial": """
#     (
#       way["landuse"="residential"](area.searchArea);
#       relation["landuse"="residential"](area.searchArea);
#       way["landuse"="industrial"](area.searchArea);
#       relation["landuse"="industrial"](area.searchArea);
#       way["landuse"="retail"](area.searchArea);
#       relation["landuse"="retail"](area.searchArea);
#       way["railway"~"rail|station|yard"](area.searchArea);
#       relation["aeroway"="aerodrome"](area.searchArea);
#     );
#     out geom;
#     """,

#     "cropland": """
#     (
#       way["landuse"="farmland"](area.searchArea);
#       relation["landuse"="farmland"](area.searchArea);
#       way["landuse"="farmland"]["crop"](area.searchArea);
#     );
#     out geom;
#     """,
    
#     "woodland": """
#     (
#       way["landuse"="forest"](area.searchArea);
#       relation["landuse"="forest"](area.searchArea);
#       way["natural"="wood"](area.searchArea);
#       relation["natural"="wood"](area.searchArea);
#     );
#     out geom;
#     """,
    
#     "shrubland": """
#     (
#       way["natural"="scrub"](area.searchArea);
#       relation["natural"="scrub"](area.searchArea);
#       way["landcover"="shrub"](area.searchArea);
#       relation["landcover"="shrub"](area.searchArea);
#     );
#     out geom;
#     """,
    
#     "grassland": """
#     (
#       way["landuse"="meadow"](area.searchArea);
#       relation["landuse"="meadow"](area.searchArea);
#       way["landuse"="grass"](area.searchArea);
#       relation["landuse"="grass"](area.searchArea);
#       way["landuse"="pasture"](area.searchArea);
#       relation["landuse"="pasture"](area.searchArea);
#     );
#     out geom;
#     """,
    
#     "bare": """
#     (
#       way["natural"="bare_rock"](area.searchArea);
#       relation["natural"="bare_rock"](area.searchArea);
#       way["landuse"="quarry"](area.searchArea);
#       relation["landuse"="quarry"](area.searchArea);
#       way["natural"="scree"](area.searchArea);
#       relation["natural"="scree"](area.searchArea);
#       way["natural"="sand"](area.searchArea);
#       relation["natural"="sand"](area.searchArea);
#     );
#     out geom;
#     """,
    
#     "water": """
#     (
#       way["natural"="water"](area.searchArea);
#       relation["natural"="water"](area.searchArea);
#       way["waterway"="river"](area.searchArea);
#       way["waterway"="stream"](area.searchArea);
#       relation["waterway"="river"](area.searchArea);
#       way["landuse"="reservoir"](area.searchArea);
#       relation["landuse"="reservoir"](area.searchArea);
#     );
#     out geom;
#     """,
    
#     "wetland": """
#     (
#       way["natural"="wetland"](area.searchArea);
#       relation["natural"="wetland"](area.searchArea);
#       way["landuse"="wetland"](area.searchArea);
#       relation["landuse"="wetland"](area.searchArea);
#       way["wetland"~"bog|fen|marsh|reedbed"](area.searchArea);
#       relation["wetland"~"bog|fen|marsh|reedbed"](area.searchArea);
#     );
#     out geom;
#     """
    
# }

# # perform queries, process and save data
# combined_gdfs = []

# for landclass, body in classes.items():
#     print(f"Downloading {landclass}...")
#     query = header + body
#     osm_json = query_overpass(query)
#     gdf = osm_to_geodf(osm_json, landclass)

#     if gdf.empty:
#         print(f"No valid geometries for {landclass}.")
#         continue
#     relevant_cols = ["id", "land_use_type", "geometry"]  # add more if needed
#     gdf = gdf[[col for col in relevant_cols if col in gdf.columns]]
#     gdf.to_file(f"../data/OSM_{landclass}.gpkg", driver="GPKG")
#     combined_gdfs.append(gdf)
#     print(f"{len(gdf)} polygons of '{landclass}'.")

# if combined_gdfs:
#     merged = gpd.GeoDataFrame(pd.concat(combined_gdfs, ignore_index=True), crs="EPSG:4326")
#     merged.to_file("../data/OSM_land_classes_combined.gpkg", driver="GPKG")
#     print(f"Combined dataset.")
# else:
#     print("No data.")

In [ ]:
# clean up variables to free memory
del lad_gdf, mvr_gdf, adreses_df, adreses_gdf, pieturas_gdf, pieturas_df# , combined_gdfs, merged

#### Data wrangling

In [ ]:
# read and process all datasets
biotopi_gdf = gpd.read_file("../data/biotopi.zip!Biotopi")
biotopi_gdf = biotopi_gdf.explode(index_parts=False)
biotopi_gdf = biotopi_gdf[~biotopi_gdf.geometry.is_empty]
biotopi_gdf = biotopi_gdf.drop_duplicates(subset='geometry')

mvr_gdf = gpd.read_file("../data/combined_mezi.gpkg")
mvr_gdf = mvr_gdf.explode(index_parts=False)
mvr_gdf = mvr_gdf[~mvr_gdf.geometry.is_empty]
mvr_gdf = mvr_gdf.drop_duplicates(subset='geometry')

lad_gdf = gpd.read_file("../data/combined_lauki.gpkg")
lad_gdf = lad_gdf.explode(index_parts=False)
lad_gdf = lad_gdf[~lad_gdf.geometry.is_empty]
lad_gdf = lad_gdf.drop_duplicates(subset='geometry')
lad_gdf = lad_gdf[pd.to_numeric(lad_gdf["PERIOD_CODE"]) == 2024]

adreses_gdf = gpd.read_file("../data/adreses.gpkg")
adreses_gdf = adreses_gdf[(adreses_gdf["FOR_BUILD"] == "N") & (adreses_gdf["APST_PAK"] != 252) & (adreses_gdf["PLAN_ADR"] == "N")]
adreses_gdf = adreses_gdf.drop_duplicates(subset='geometry')

pieturas_gdf = gpd.read_file("../data/pieturas.gpkg")
pieturas_gdf = pieturas_gdf.drop_duplicates(subset='geometry')
pieturas_gdf = pieturas_gdf[~pieturas_gdf.geometry.is_empty]

# osm_gdf = gpd.read_file("../data/OSM_land_classes_combined.gpkg").to_crs("EPSG:3059")

In [ ]:
# define land use categories based on data attributes
# wetland
wetland_bio_gdf = biotopi_gdf[biotopi_gdf["CODE_EC"].isin(["7110*", "7120", "7140", "7150", "7210*"])]

# water
water_bio_gdf = biotopi_gdf[biotopi_gdf["CODE_EC"].isin(["1170", "3130", "3140", "3150", "3160", "3260"])]

# bare
bare_bio_gdf = biotopi_gdf[biotopi_gdf["CODE_EC"].isin(["1110", "1230", "1310", "2110", "2120", "2170", "2330"])]
bare_mvr_gdf = mvr_gdf[(mvr_gdf["zkat"].astype(int) == 10) & (mvr_gdf["p_cirg"].astype(int) == 2023) & (mvr_gdf["p_cirp"].astype(int) == 11)]
bare_gdf = gpd.GeoDataFrame(pd.concat([bare_bio_gdf, bare_mvr_gdf], ignore_index=True), crs='EPSG:3059')

# grassland
grassland_bio_gdf = biotopi_gdf[biotopi_gdf["CODE_EC"].isin(["6100", "6110*", "6120*", "6210", "6230*", "6270*", "6410", "6450", "6510"])]
grassland_lad_gdf = lad_gdf[pd.to_numeric(lad_gdf["PRODUCT_CODE"], errors="coerce").isin([710, 713, 720, 760])]
grassland_gdf = gpd.GeoDataFrame(pd.concat([grassland_bio_gdf, grassland_lad_gdf], ignore_index=True), crs='EPSG:3059')

# shrubland
shrubland_bio_gdf = biotopi_gdf[biotopi_gdf["CODE_EC"].isin(["2320", "4030", "5130"])]

# woodland
woodland_bio_gdf = biotopi_gdf[biotopi_gdf["CODE_EC"].isin(["2180", "9010*", "9020*", "9050", "9060", "9070", "9080*", "9160", "9180*", "91E0*", "91F0", "91T0"])]
woodland_mvr_gdf = mvr_gdf[(mvr_gdf["zkat"] == 10) & (mvr_gdf["a10"].astype(int) > 5)]
woodland_gdf = gpd.GeoDataFrame(pd.concat([woodland_bio_gdf, woodland_mvr_gdf], ignore_index=True), crs='EPSG:3059')

# cropland
cropland_lad_gdf = lad_gdf[~pd.to_numeric(lad_gdf["PRODUCT_CODE"], errors="coerce").isin([710, 713, 720, 760])] 

# artificial
artificial_gdf = gpd.GeoDataFrame(pd.concat([adreses_gdf, pieturas_gdf], ignore_index=True), crs='EPSG:3059')

In [ ]:
# combine all polygon land use types into a single GeoDataFrame for point filtering
land_use_gdf = gpd.GeoDataFrame(columns=['geometry', 'land_use_type'], crs='EPSG:3059')
land_use_gdf = gpd.GeoDataFrame(
    pd.concat([
        wetland_bio_gdf.assign(land_use_type='wetland'),
        water_bio_gdf.assign(land_use_type='water'),
        bare_gdf.assign(land_use_type='bare'),
        grassland_gdf.assign(land_use_type='grassland'),
        shrubland_bio_gdf.assign(land_use_type='shrubland'),
        woodland_gdf.assign(land_use_type='woodland'),
        cropland_lad_gdf.assign(land_use_type='cropland')
    ], ignore_index=True),
    crs='EPSG:3059'
)

land_use_gdf['geometry'] = land_use_gdf.geometry.buffer(5)

In [ ]:
# define a function to buffer and sample points from the GeoDataFrame
def buffer_and_sample(gdf):
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.buffer(-15)
    
    gdf = gdf.explode(index_parts=False)
    gdf = gdf[~gdf.geometry.is_empty]
    
    gdf["geometry"] = gdf.geometry.representative_point()
    return gdf

# Wetland
wetland_sample_points = buffer_and_sample(wetland_bio_gdf)

# Water
water_sample_points = buffer_and_sample(water_bio_gdf)

# Bare
bare_sample_points = buffer_and_sample(bare_gdf)

# Grassland
grassland_sample_points = buffer_and_sample(grassland_gdf)

# Shrubland
shrubland_sample_points = buffer_and_sample(shrubland_bio_gdf)

# Woodland
woodland_sample_points = buffer_and_sample(woodland_gdf)

# Cropland
cropland_sample_points = buffer_and_sample(cropland_lad_gdf)

# OSM
# osm_sample_points = buffer_and_sample(osm_gdf)
# joined_osm = gpd.sjoin(osm_sample_points, land_use_gdf, predicate="intersects", how="left")
# osm_sample_points = joined_osm[joined_osm.index_right.isna()].drop(columns=["index_right"])
# osm_sample_points['land_use_type'] = osm_sample_points['land_use_type_left']

In [ ]:
# define relevant columns and combine all sample points into a single GeoDataFrame, filtering out points that intersect with other land use types, save to CSV
relevant_columns = ['land_use_type', 'lon', 'lat', 'geometry']
combined_sample_points = gpd.GeoDataFrame(columns=relevant_columns, crs='EPSG:4326')

for land_use_type, sample_points in zip(
    ['wetland', 'water', 'bare', 'grassland', 'shrubland', 'woodland', 'cropland', 'artificial'],
    [wetland_sample_points, water_sample_points, bare_sample_points, grassland_sample_points, shrubland_sample_points, woodland_sample_points, cropland_sample_points, artificial_gdf]
):
    n_samples = min(2000, len(sample_points))
    if n_samples == 0:
        continue
    # osm_sampled = osm_sample_points[osm_sample_points['land_use_type'] == land_use_type]
    # sample_points = pd.concat([sample_points, osm_sampled], ignore_index=True)
    
    sampled = sample_points.sample(n=n_samples, random_state=42)
    sampled = sampled.to_crs("EPSG:3059")

    other_polygons = land_use_gdf[land_use_gdf['land_use_type'] != land_use_type]

    joined = sampled.sjoin(other_polygons, how='left', predicate='within')

    clean_sampled = joined[joined['index_right'].isna()].drop(columns=['index_right'])

    clean_sampled = clean_sampled.to_crs("EPSG:4326")

    clean_sampled['land_use_type'] = land_use_type

    clean_sampled['lon'] = clean_sampled.geometry.x
    clean_sampled['lat'] = clean_sampled.geometry.y
    clean_sampled = clean_sampled[relevant_columns]

    combined_sample_points = pd.concat([combined_sample_points, clean_sampled], ignore_index=True)



combined_sample_points.to_csv("../data/combined_sample_points_fin.csv", index=False)
